# Configuración invial spark en GoogleColab

In [1]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null # Instalar SDK java 17

In [2]:
!wget -q https://downloads.apache.org/spark/spark-4.0.2/spark-4.0.2-bin-hadoop3.tgz # Descargar Spark 4.0.1

In [3]:
!tar xf spark-4.0.2-bin-hadoop3.tgz # Descomprimir la version de Spark

In [4]:
import os # Establecer las variables de entorno

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.0.2-bin-hadoop3"

In [5]:
!pip install -q findspark # Instalar la librería findspark

In [6]:
!pip install -q pyspark # Instalar pyspark

In [7]:
# Verificar la instalación

import findspark

findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()

In [8]:
# Probando la sesión de Spark

df = spark.createDataFrame([{"Hola": "Mundo"} for x in range(10)]) # DF random para ver si funciona

df.show(10, False)

+-----+
|Hola |
+-----+
|Mundo|
|Mundo|
|Mundo|
|Mundo|
|Mundo|
|Mundo|
|Mundo|
|Mundo|
|Mundo|
|Mundo|
+-----+



# Spark Session

Spark session es un punto de partda unificado que permite crear grandes dataframes, leer fuentes de datos, acceder a metadatos de catálogos y emitir consultas Spark SQL.

In [14]:
# Creamos la sesión de Spark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("CursoPySpark").getOrCreate() # master() si es en un cluster

In [16]:
# spark

# RDD: Resilient Distributed Dataset

Son una colección de elementos particionados a través de los clusters que pueden ser trabajadas en paralelo.

Hay tres elementos a tener en cuenta de los RDD:

- Dependencias: lista que le indica a spark como se construye un RDD con sus entradas. Aporta resilencia ante fallas

- Particiones: se divide el trabajo paralizando el calculo entre ejecutores.

- Calculo: iterador para datos que se almacenaran en el RDD.

HDFS: Hadoop Distributed File System, tienen una correspondencia con las particiones.

## Diferentes formas de crear un RDD

In [20]:
sc = spark.sparkContext # Punto de entrada para crear RDD

In [23]:
# Crear un RDD vacio

rdd_vacio = sc.emptyRDD
rdd_vacio

<bound method SparkContext.emptyRDD of <SparkContext master=local[*] appName=pyspark-shell>>

In [24]:
# RDD paralelizado

rdd_paralelizado = sc.parallelize([], 3) # Lista vacia, particiones

rdd_paralelizado.getNumPartitions()

3

In [29]:
# RDD con datos paralelizado

rdd_condatos_par = sc.parallelize([1, 2, 3, 4, 5])
rdd_condatos_par

ParallelCollectionRDD[12] at readRDDFromFile at PythonRDD.scala:297

In [30]:
rdd_condatos_par.collect() # Me trae los elementos

[1, 2, 3, 4, 5]

In [33]:
# Crear un RDD desde un archivo de texto

rdd_texto = sc.textFile("/content/drive/MyDrive/RDD_FILES/rdd_source.txt")

rdd_texto.collect()

['Así podemos crear', 'un RDD desde un', 'archivo de texto!!!']

In [32]:
# RDD con el texto completo

rdd_texto_completo = sc.wholeTextFiles("/content/drive/MyDrive/RDD_FILES/rdd_source.txt")

rdd_texto_completo.collect()

[('file:/content/drive/MyDrive/RDD_FILES/rdd_source.txt',
  'Así podemos crear\nun RDD desde un\narchivo de texto!!!')]

In [37]:
# RDD Suma

rdd_suma = rdd_condatos_par.map(lambda x: x + 1)
rdd_suma.collect()

[2, 3, 4, 5, 6]

In [42]:
# DataFarme

df = spark.createDataFrame([(1, "Martin"), (2, "Juancho")], ["Id", "Nombre"])
df.show()


+---+-------+
| Id| Nombre|
+---+-------+
|  1| Martin|
|  2|Juancho|
+---+-------+



In [43]:
# RDD con DataFrame

rdd_df = df.rdd
rdd_df.collect()

[Row(Id=1, Nombre='Martin'), Row(Id=2, Nombre='Juancho')]

# Transformaciones de un RDD

Los RDD son *inmutables*, por cada operación que genere un cambio se crea un nuevo.

Las dos *operaciones* que se pueden realizar son las siguientes:

- **Transformaciones:** cambian los elementos en le RDD, dividen los elementos de entrada, filtran elementos, realizan calculos.

- **Acciones:** se estudiarán luego.



## Tipos de transformaciones

Hay cuatro tipos de transformaciones:

- **Generales:** funciones de agregación
- **Matemáticas o estadísticas:** normalización, escalamiento, muestreo
- **Conjunto o relacionales:** uniones de conjunto de datos, etc. Similares de las de SQL pero con cambios en la sintaxis.
- **Estructura de datos:** operan en las estructuras de datos subyacentes tales como las particiones.

Algunas transformaciones **de conjunto o relacionales** son:

- cogroup
- join
- subtractByKey
- fullOuterJoin
- leftOuterJoin
- rigthOuterJoin

Algunas transformaciones **de estructuras de datos** son:

- partitionBy
- repartition
- zipwithIndex
- coalcase


### Función MAP

In [48]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [50]:
sc = spark.sparkContext
rdd = sc.parallelize([1, 2, 3, 4, 5])

In [51]:
rdd_resta = rdd.map(lambda x: x - 1)
rdd_resta.collect()

[0, 1, 2, 3, 4]

In [52]:
rdd_par = rdd.map(lambda x: x % 2 == 0)
rdd_par.collect()

[False, True, False, True, False]

In [54]:
rdd_texto = sc.parallelize(["martin", "gonzales", "fulanito"])
rdd_mayuscula = rdd_texto.map(lambda x:x.upper())
rdd_mayuscula.collect()

['MARTIN', 'GONZALES', 'FULANITO']

In [55]:
rdd_hola = rdd_texto.map(lambda x: "Hola " + x)
rdd_hola.collect()

['Hola martin', 'Hola gonzales', 'Hola fulanito']

### Función FLATMAP

In [56]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

sc = spark.sparkContext

In [57]:
rdd = sc.parallelize([1, 2, 3, 4, 5])

In [61]:
rdd_cuadrado = rdd.flatMap(lambda x: (x, x ** 2)) # Sin el flatMap collect() retornaria una tupla por cada elemento al cuadrado
rdd_cuadrado.collect()

[1, 1, 2, 4, 3, 9, 4, 16, 5, 25]

In [62]:
rdd_texto = sc.parallelize(["martin", "gonzales", "fulanito"])

rdd_mayuscula = rdd_texto.flatMap(lambda x: (x, x.upper())) # En este caso retorna una sola lista con la cadena y su mayuscula en una misma lista
rdd_mayuscula.collect()

['martin', 'MARTIN', 'gonzales', 'GONZALES', 'fulanito', 'FULANITO']

### Función FILTER

In [63]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

sc = spark.sparkContext

In [65]:
rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [67]:
# Valores pares

rrd_par = rdd.filter(lambda x: x % 2 == 0)
rrd_par.collect()

[2, 4, 6, 8]

In [68]:
# Valores impares

rdd_impar = rdd.filter(lambda x: x % 2 != 0)
rdd_impar.collect()

[1, 3, 5, 7, 9]

In [71]:
# Cadenas que empiecen con K

rdd_texto = sc.parallelize(['jose', 'juaquin', 'juan', 'lucia', 'karla', 'katia'])

rdd_k = rdd_texto.filter(lambda x: x.startswith('k'))

rdd_k.collect()

['karla', 'katia']

In [70]:
# Comienza con "j" termina con "u"

rdd_filtro = rdd_texto.filter(lambda x: x.startswith('j') and x.find('u') == 1)

rdd_filtro.collect()

['juaquin', 'juan']

### Función COALESCE

Combina las particiones en las índicadas

In [75]:
rdd = sc.parallelize([1, 2, 3.4, 5], 10)
rdd.getNumPartitions()

10

In [81]:
rdd = rdd.coalesce(5) # Recordemos que los RDD son inmutables, debemos asignarlo al mismo o a otro si queremos cambiar su estructura
rdd.getNumPartitions()

5

### Función REPARTITION

Reparticiona las particiones en más o menos particiones de salida según se indique o sea necesario, es más costosa de COALESCE.

In [82]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [84]:
rdd = sc.parallelize([1, 2, 3, 4, 5], 3)
rdd.getNumPartitions()

3

In [85]:
# Cambiamos la cantidad de particiones

rdd7 = rdd.repartition(7)
rdd7.getNumPartitions()

7

### Función reduceByKey

Agrupa los datos por clave valor sumando, en este caso, el valor del segundo elemento de la tupla. Se puede multplicar también o aplicar más operaciones incluidas técnicas de hashing.

In [86]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [87]:
rdd = sc.parallelize(
    [('casa', 2),
     ('parque', 1),
     ('que', 5),
     ('casa', 1),
     ('escuela', 2),
     ('casa', 1),
     ('que', 1)]
)

rdd.collect()


[('casa', 2),
 ('parque', 1),
 ('que', 5),
 ('casa', 1),
 ('escuela', 2),
 ('casa', 1),
 ('que', 1)]

In [88]:
rdd_reduciodo = rdd.reduceByKey(lambda x,y: x + y)
rdd_reduciodo.collect()

[('casa', 4), ('parque', 1), ('que', 6), ('escuela', 2)]

# Acciones

Las **acciones** son operaciones que activan los calculos. Hasta que estas no se producen la ejecución se ejecuta en forma de grafo.

## Tipos de acciones

- Las que ocurren en el **driver**: operaciones remotas que retornan su resultado al controlador.
- Las **distribuidas**: se ejecutan en los nodos del cluster.

### Función REDUCE

Realiza una operación con la información de las particiones y la envia al driver, no como una unica unidad sino como un valor

In [89]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [93]:
rdd = sc.parallelize([1, 2, 3, 4, 5])

In [94]:
rdd.reduce(lambda x,y: x + y)

15

### Función COUNT

Cuenta el número de elementos del RDD y lo envía al controlador

In [98]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [100]:
rdd = sc.parallelize(['m', 'a', 'r', 't', 'i', 'n'])

In [101]:
rdd.count()

6

### Función COLLECT

Recopila los datos de RDD y los envía al controlador como datos, no como particiones.

In [102]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [103]:
rdd = sc.parallelize("Hola Mundo Apache".split(" "))

In [104]:
rdd.collect()

['Hola', 'Mundo', 'Apache']

### Función take, max, min y saveAsTextFile

**Take** toma los primeros n elementos indicados.
**Max** toma el valor máximo
**Min** toma el valor minimo
**saveAsTextFile** guardará la cadena en un texto

Hay muchas más parecidas a las funciones de agregación SQL o variantes de estas.

In [105]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [106]:
rdd = sc.parallelize("Elementos de la lista".split(" "))
rdd.take(2)

['Elementos', 'de']